In [1]:
import time
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score
import great_expectations as ge
from sklearn.preprocessing import StandardScaler

In [13]:
def process_stream_in_batches(data_path, chunksize=100000):
    print(f"\n==================================================")
    print(f"STREAMING PIPELINE (CHUNK SIZE: {chunksize:,}) ON: {data_path}")
    print(f"==================================================")
    
    t_start_total = time.time()
    
    # Accumulated prediction & target lists across streams
    all_y_true = []
    all_y_pred = []
    
    total_repair_time = 0.0
    total_validation_time = 0.0
    batch_count = 0
    
     # --- Periodic retraining state ---
    RETRAIN_INTERVAL = 10          # refit every 10 batches
    model = None                   # persistent model across batches
    scaler = None                  # persistent scaler across batches
    refit_count = 0                # count how many times we refit
    
    # 1. Stream file in micro-batches using pandas chunking
    for chunk in pd.read_csv(data_path, chunksize=chunksize):
        batch_count += 1
        
        # Filter relevant transaction types
        df_filtered = chunk[chunk['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
        if batch_count <= 2:
            print(f"[DEBUG] Batch {batch_count}: {len(chunk)} rows → {len(df_filtered)} after filter")
            
            
        # ----------------------------------------------------
        # STAGE 1: DECLARATIVE VALIDATION GATEWAY
        # ----------------------------------------------------
        t_start_valid = time.time()
        context = ge.get_context(mode="ephemeral")
    
        datasource = context.data_sources.add_pandas(
        name="in_memory_datasource"
    )
        data_asset = datasource.add_dataframe_asset(name="paysim_asset")
    
        # Step 1: Create a Batch Definition (no DataFrame here)
        batch_definition = data_asset.add_batch_definition_whole_dataframe(
            name="paysim_batch_definition"
        )
    
        # Step 2: Get the Batch by passing the DataFrame HERE
        batch = batch_definition.get_batch(
            batch_parameters={"dataframe": df_filtered}
        )
    
        # Step 3: Create the suite and validator from the Batch
        suite = context.suites.add(
            ge.ExpectationSuite(name="paysim_suite")
        )
        validator = context.get_validator(
            batch=batch,
            expectation_suite=suite
        )
            
        target_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
        for col in target_cols:
            validator.expect_column_values_to_not_be_null(column=col)
        validator.expect_column_values_to_be_between(column='amount', min_value=0)
        
        validation_results = validator.validate()
        t_end_valid = time.time()
        
        t_valid_duration = t_end_valid - t_start_valid
        total_validation_time += t_valid_duration
        validation_passed = validation_results["success"]
        
        print(f"[Stage 1] Validation Status: {'PASSED' if validation_passed else 'FAILED (Corruptions Detected)'}")
        print(f"[Stage 1] Validation Latency (T_valid): {t_valid_duration:.4f} seconds")
        
        # ----------------------------------------------------
        # STAGE 2: PROBABILISTIC REPAIR LAYER (HoloClean Logic)
        # ----------------------------------------------------
        t_r_start = time.time()
        df_repaired = df_filtered.copy()
        
        if not validation_passed:
            # Relational Denial Constraints
            mask_old = df_repaired['oldbalanceOrg'].isna() & df_repaired['newbalanceOrig'].notna() & df_repaired['amount'].notna()
            df_repaired.loc[mask_old, 'oldbalanceOrg'] = df_repaired.loc[mask_old, 'newbalanceOrig'] + df_repaired.loc[mask_old, 'amount']
            
            mask_new = df_repaired['newbalanceOrig'].isna() & df_repaired['oldbalanceOrg'].notna() & df_repaired['amount'].notna()
            df_repaired.loc[mask_new, 'newbalanceOrig'] = df_repaired.loc[mask_new, 'oldbalanceOrg'] - df_repaired.loc[mask_new, 'amount']
            
            mask_amt = df_repaired['amount'].isna() & df_repaired['oldbalanceOrg'].notna() & df_repaired['newbalanceOrig'].notna()
            df_repaired.loc[mask_amt, 'amount'] = df_repaired.loc[mask_amt, 'oldbalanceOrg'] - df_repaired.loc[mask_amt, 'newbalanceOrig']
            
            # Fallback imputations
            for col in target_cols:
                if df_repaired[col].isna().any():
                    df_repaired[col] = df_repaired[col].fillna(0)     
        t_r_end = time.time()
        total_repair_time += (t_r_end - t_r_start)
        
        
        # ----------------------------------------------------
        # STAGE 3: DOWNSTREAM MODELING (Periodic Retraining)
        # ----------------------------------------------------
        # Domain Feature Engineering
        df_repaired['balance_diff_orig'] = (
            df_repaired['oldbalanceOrg'] - df_repaired['amount'] - df_repaired['newbalanceOrig']
        )
        # Fraud indicator: Origin account completely emptied
        df_repaired['orig_zero_after'] = ((df_repaired['oldbalanceOrg'] > 0) & (df_repaired['newbalanceOrig'] == 0)).astype(int)
        
        # Fraud indicator: Destination account balance unchanged despite transfer
        df_repaired['dest_no_change'] = ((df_repaired['amount'] > 0) & (df_repaired['oldbalanceDest'] == df_repaired['newbalanceDest'])).astype(int)

        feature_cols = [
            'amount', 'oldbalanceOrg', 'newbalanceOrig', 
            'oldbalanceDest', 'newbalanceDest', 'balance_diff_orig',
            'orig_zero_after', 'dest_no_change'
        ]
        
        X = df_repaired[feature_cols].fillna(0)
        y = df_repaired['isFraud']

        # Retrain only when: (a) first batch, or (b) interval reached
        if model is None or (batch_count - 1) % RETRAIN_INTERVAL == 0:
            # Fresh scaler + model
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X)

            model = IsolationForest(
                n_estimators=100,
                contamination=max(y.mean(), 0.001),
                random_state=42,
                n_jobs=-1
            )
            model.fit(X_scaled)
            refit_count += 1
            print(f"[Stage 3] Model REFIT at batch {batch_count} (refit #{refit_count})")

            # Score the batch we just trained on
            preds = model.predict(X_scaled)
        else:
            # Use frozen scaler and model
            X_scaled = scaler.transform(X)   # transform, NOT fit_transform
            preds = model.predict(X_scaled)

        preds_binary = (preds == -1).astype(int)

        all_y_true.extend(y.tolist())
        all_y_pred.extend(preds_binary.tolist())
    t_end_total = time.time()
    
    # Calculate Overall Stream Performance Metrics
    overall_precision = precision_score(all_y_true, all_y_pred, pos_label=1, zero_division=0)
    overall_recall = recall_score(all_y_true, all_y_pred, pos_label=1, zero_division=0)
    overall_f1 = f1_score(all_y_true, all_y_pred, pos_label=1, zero_division=0)
    total_latency = t_end_total - t_start_total
    
    print(f"\n\n=== STREAMING EVALUATION RESULTS ===")
    print(f"Retraining Strategy: Periodic (every {RETRAIN_INTERVAL} batches)")
    print(f"Total Batches Processed: {batch_count}")
    print(f"Total Model Refits: {refit_count}")
    print(f"Total Stream Time (Delta T): {total_latency:.4f} seconds")
    print(f"Validation Overhead: {total_validation_time:.4f} seconds")
    print(f"Repair Overhead (Delta T_r): {total_repair_time:.4f} seconds")
    print(f"Overall Stream Precision: {overall_precision:.4f}")
    print(f"Overall Stream Recall: {overall_recall:.4f}")
    print(f"Overall Stream F1-Score: {overall_f1:.4f}")

if __name__ == "__main__":
    process_stream_in_batches('Dataset.csv', chunksize=100000)


STREAMING PIPELINE (CHUNK SIZE: 100,000) ON: Dataset.csv
[DEBUG] Batch 1: 100000 rows → 39315 after filter


Calculating Metrics: 100%|██████████| 21/21 [00:00<00:00, 423.44it/s]


[Stage 1] Validation Status: PASSED
[Stage 1] Validation Latency (T_valid): 0.7312 seconds
[Stage 3] Model REFIT at batch 1 (refit #1)
[DEBUG] Batch 2: 100000 rows → 44009 after filter


Calculating Metrics: 100%|██████████| 21/21 [00:00<00:00, 473.06it/s]


[Stage 1] Validation Status: PASSED
[Stage 1] Validation Latency (T_valid): 2.7952 seconds


...

[Stage 1] Validation Status: PASSED
[Stage 1] Validation Latency (T_valid): 0.4766 seconds


Calculating Metrics: 100%|██████████| 21/21 [00:00<00:00, 573.53it/s]


[Stage 1] Validation Status: PASSED
[Stage 1] Validation Latency (T_valid): 0.4659 seconds


Calculating Metrics: 100%|██████████| 21/21 [00:00<00:00, 497.87it/s]


[Stage 1] Validation Status: PASSED
[Stage 1] Validation Latency (T_valid): 0.4932 seconds


=== STREAMING EVALUATION RESULTS ===
Retraining Strategy: Periodic (every 10 batches)
Total Batches Processed: 64
Total Model Refits: 7
Total Stream Time (Delta T): 81.7858 seconds
Validation Overhead: 32.3592 seconds
Repair Overhead (Delta T_r): 0.4499 seconds
Overall Stream Precision: 0.1277
Overall Stream Recall: 0.1953
Overall Stream F1-Score: 0.1544


In [14]:
process_stream_in_batches('paysim_noise_5.csv', chunksize=100000)
process_stream_in_batches('paysim_noise_10.csv', chunksize=100000)
process_stream_in_batches('paysim_noise_20.csv', chunksize=100000)


STREAMING PIPELINE (CHUNK SIZE: 100,000) ON: paysim_noise_5.csv
[DEBUG] Batch 1: 100000 rows → 100000 after filter


Calculating Metrics: 100%|██████████| 21/21 [00:00<00:00, 244.40it/s]


[Stage 1] Validation Status: FAILED (Corruptions Detected)
[Stage 1] Validation Latency (T_valid): 0.6142 seconds
[Stage 3] Model REFIT at batch 1 (refit #1)
[DEBUG] Batch 2: 100000 rows → 100000 after filter


Calculating Metrics: 100%|██████████| 21/21 [00:00<00:00, 306.32it/s]


[Stage 1] Validation Status: FAILED (Corruptions Detected)
[Stage 1] Validation Latency (T_valid): 0.5942 seconds


...

[Stage 1] Validation Status: FAILED (Corruptions Detected)
[Stage 1] Validation Latency (T_valid): 0.5600 seconds


Calculating Metrics: 100%|██████████| 21/21 [00:00<00:00, 406.98it/s]


[Stage 1] Validation Status: FAILED (Corruptions Detected)
[Stage 1] Validation Latency (T_valid): 0.7491 seconds


Calculating Metrics: 100%|██████████| 21/21 [00:00<00:00, 295.92it/s]


[Stage 1] Validation Status: FAILED (Corruptions Detected)
[Stage 1] Validation Latency (T_valid): 0.7098 seconds


=== STREAMING EVALUATION RESULTS ===
Retraining Strategy: Periodic (every 10 batches)
Total Batches Processed: 28
Total Model Refits: 3
Total Stream Time (Delta T): 54.3624 seconds
Validation Overhead: 20.3519 seconds
Repair Overhead (Delta T_r): 1.0776 seconds
Overall Stream Precision: 0.0443
Overall Stream Recall: 0.0576
Overall Stream F1-Score: 0.0501
